# AgentCore

Evaluate any agent deployed with [AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-invoke-agent.html) by invoking its runtime endpoint and running UAEF metrics on the response.

AgentCore agents are identified by their **Runtime ARN**. The adapter calls `bedrock-agentcore:invoke_agent_runtime`.

**Auth modes supported:**
- **SigV4 (default)** — uses the boto3 SDK and works when the agent runtime is configured for IAM auth.
- **OAuth / JWT** — if your agent runtime is configured with a JWT authorizer, pass a `BEARER_TOKEN`. The adapter makes a raw HTTPS request instead of using the SDK.

This notebook shows two flows:
1. **Single evaluation** — invoke the deployed agent once, transform the trace, and evaluate.
2. **Batch evaluation** — send every question from a ground-truth spreadsheet to the agent and evaluate all responses.

#### Authenticate with AWS

In [ ]:
# !pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv(".env", override=True)
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *

for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()

#### Choose the set of metrics for evaluation

In [ ]:
metrics = all_metrics
print("You've chosen the following metrics for evaluating the agent deployed with AgentCore:")
for m in metrics:
    print(f"  - {m}")

#### Select AWS region

In [ ]:
region = "us-east-1"
print(f"AWS Region: {region}")

#### Install UAEF
Go into root of directory and run `pip install -e .`

In [ ]:
pip install -e ../.

In [ ]:
pip install aiobotocore

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone
print("✓ UAEF imported successfully!")

# AgentCore with Only 1 Agent Deployed

## Configure AgentCore Runtime

Create `.env` file with the following:
- `RUNTIME_ARN` for AgentCore runtime ARN 


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env")        # loads .env.agentcore (AgentCore config)

runtime = os.getenv("RUNTIME_ARN")
print(f"Agent Runtime ARN: {runtime}")



#### ONLY run the following cells if you have AgentCore runtime configured with JWT authentication (bearer token): 
Skip these cells if you're using SigV4 authentication.

In [ ]:
import boto3

def get_cognito_token(user_pool_id, client_id, username, password):
    """Authenticate with Cognito and return an access token."""
    client = boto3.client("cognito-idp", region_name="us-east-1")
    response = client.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={
            "USERNAME": username,
            "PASSWORD": password,
        },
    )
    return response["AuthenticationResult"]["AccessToken"]


Retrieve your Bearer Token using Cognito credentials:

In [ ]:
os.environ["RUNTIME_BEARER_TOKEN"] = get_cognito_token(
    user_pool_id="<enter-your-cognito-user-pool-id>",
    client_id="<enter-your-client-ID>",
    username="<enter-your-username>",
    password="<enter-your-password>",
)


Set up bearer token in your `.env` file:

In [ ]:
from dotenv import set_key

# Write to .env file (creates or updates the RUNTIME_BEARER_TOKEN line)
set_key(".env", "RUNTIME_BEARER_TOKEN", os.environ["RUNTIME_BEARER_TOKEN"])
RUNTIME_BEARER_TOKEN = os.environ["RUNTIME_BEARER_TOKEN"]
print("✓ Bearer token written to .env and loaded into environment")

### Proceed with the following cells for **both** SigV4 & JWT authentication:

### Pass a single query through AgentCore adapter

Confirm Runtime ARN and region:

In [ ]:
print(f"AgentCore Runtime ARN for single-agent solution:\n\n {runtime})")

region = "us-east-1"
print(f"\n\nAWS Region: {region}")

Load AgentCore Adapter to run evaluation

In [ ]:
from uaef.adapters import AgentCoreAdapter

# Set up AgentCore adapter
adapter = AgentCoreAdapter()
print(f"✓ AgentCoreAdapter ready")


## Single Evaluation

#### Step 1 — Invoke the deployed agent and collect the response

In [ ]:
user_input = "Compare the trend direction for the period ending 2022-10-14 versus the period ending 2023-02-02. Which period had a higher 20-day SMA?"

# Invoke agent 
raw_data = adapter.invoke(
    agent_runtime_arn=runtime,
    user_input=user_input,
    agent_version="DEFAULT",
    log_stream_name="otel-rt-logs",
    region=region,
    # bearer_token=RUNTIME_BEARER_TOKEN, # only enable when using JWT auth
)


print(f"✓ Agent responded in {raw_data['latency']:.2f}s")
print(f"\nAgent Response: {raw_data['final_response'][:500]}")

#### Step 2 — Transform to canonical AgentTrace

In [ ]:

agent_trace = adapter.transform_to_canonical(raw_data)
print(f"✓ Transformed AgentCore output to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")

if agent_trace.input_tokens:
    print(f"  Input Tokens: {agent_trace.input_tokens}")
if agent_trace.output_tokens:
    print(f"  Output Tokens: {agent_trace.output_tokens}")


# Print extracted messages
for msg in agent_trace.messages:
    print(f"\n  [{msg.role.value}]: {msg.content[:200]}")


#### Step 3 — Evaluate

In [ ]:
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone
from uaef.models import GroundTruth, ToolCall

ground_truth = GroundTruth(
    expected_output="Trend direction: downtrend vs uptrend. 20-day SMA: 116.14 vs 96.58...",
    expected_tool_calls=[
        ToolCall(name="detect_trend", arguments={"end_date": "2022-10-14"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="detect_trend", arguments={"end_date": "2023-02-02"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="calculate_moving_average", arguments={"end_date": "2022-10-14"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="calculate_moving_average", arguments={"end_date": "2023-02-02"}, timestamp=datetime.now(timezone.utc)),
    ],
)


result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
)
print(f"\n{'='*50}")
print("AGENTCORE — EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        if metric.score is not None:
            print(f"  {metric.metric_name}: {metric.score:.2f}")
        else:
            print(f"  {metric.metric_name}: N/A (failed)")


## Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation by sending queries from Ground Truth to the deployed agent.

#### Load Ground Truth Q & A

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime, timezone
from uuid import uuid4

from uaef.models import GroundTruth, ToolCall
from uaef.models.message import MessageRole
from uaef.evaluation.multi_agent import MultiAgentEvaluator

# Load ground truth data from Excel
excel_path = "data/singleagent_agentcore_ground_truth.csv"
df_gt = pd.read_csv(excel_path)
gt_json = df_gt.to_dict(orient="records")

print(f"✓ Loaded {len(gt_json)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df_gt.columns)}")
df_gt.head()


#### Invoke, Transform, and Evaluate

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import pandas as pd
from datetime import datetime
from uaef.adapters.agentcore import AgentCoreAdapter
from uaef.api import evaluate
from uaef.models.ground_truth import GroundTruth
from uaef.models.message import MessageRole

adapter = AgentCoreAdapter()
results = []
traces = []


In [ ]:


for idx, row in df_gt.iterrows():
    query = row["question"]
    expected_output = row["expected_output"]
    print(f"\n[{idx+1}/{len(df_gt)}] {query[:80]}...")

    # Parse expected_tool_calls from CSV into ToolCall objects
    expected_tool_calls = []
    if pd.notna(row.get("expected_tool_calls", "")):
        try:
            # If stored as JSON string in CSV
            raw_tools = json.loads(str(row["expected_tool_calls"]))
            if isinstance(raw_tools, list):
                for t in raw_tools:
                    if isinstance(t, dict):
                        expected_tool_calls.append(ToolCall(
                            name=t.get("name", ""),
                            arguments=t.get("arguments", t.get("args", t.get("input", {}))),
                            timestamp=datetime.now(timezone.utc),
                        ))
                    elif isinstance(t, str):
                        # Just tool names
                        expected_tool_calls.append(ToolCall(name=t, arguments={}))
        except (json.JSONDecodeError, TypeError):
            # Fallback: comma-separated tool names
            for name in str(row["expected_tool_calls"]).split(","):
                name = name.strip()
                if name:
                    expected_tool_calls.append(ToolCall(name=name, arguments={}, timestamp=datetime.now(timezone.utc)))

    # 1. Invoke
    raw_data = AgentCoreAdapter.invoke(
        agent_runtime_arn=runtime,
        user_input=query,
        agent_version="DEFAULT",
        log_stream_name="otel-rt-logs",
        region=region,
        # bearer_token=RUNTIME_BEARER_TOKEN, # only enable when using JWT auth
    )

    # 2. Transform
    agent_trace = adapter.transform_to_canonical(raw_data)
    traces.append(agent_trace)

    # 3. Evaluate
    ground_truth = GroundTruth(expected_output=expected_output, expected_tool_calls=expected_tool_calls)
    result = evaluate(
        trace=agent_trace,
        ground_truth=ground_truth,
        metrics=metrics,
    )
    results.append(result)

    print(f"  Score: {result.overall_score:.2f} | Tools: {len(agent_trace.tool_calls)}")


#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="single_agentcore_batch_results")


# Sample MultiAgent AgentCore - Single Query Evaluation

In [ ]:
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole
from uaef.adapters import AgentCoreAdapter
from uaef.models.multi_agent_trace import MultiAgentTrace

from datetime import datetime, timezone
from uuid import uuid4
import json
import os
import re
import boto3
adapter = AgentCoreAdapter()
print("✓ UAEF imported successfully!")


Create a `.env` file and populate it with the both Runtime ARN and AWS credentials:

-  `runtime_arn` value for your multi-agentic solution
- `AWS_ACCESS_KEY_ID` 
- `AWS_SECRET_ACCESS_KEY`
- `AWS_SESSION_TOKEN`

In [ ]:
from dotenv import load_dotenv


load_dotenv(".env", override=True)

def discover_agents():
    """
    Auto-discover AgentCore agents from environment variables.
    
    Scans ALL env vars for values that look like AgentCore runtime ARNs
    (arn:aws:bedrock-agentcore:*:runtime/*). The env var name becomes
    the agent's display name.
    
    For each discovered ARN env var (e.g. HR_AGENT_RUNTIME_ARN), it also
    looks for a matching bearer token env var by replacing _ARN with _BEARER_TOKEN
    or _RUNTIME_ARN with _BEARER_TOKEN, and a runtime ID var by replacing
    _ARN with _ID.
    
    Examples of env vars that will be discovered:
        CONCIERGE_AGENT_RUNTIME_ARN=arn:aws:bedrock-agentcore:...
        AGENT_RUNTIME_ARN=arn:aws:bedrock-agentcore:...
    """
    arn_pattern = re.compile(r"^arn:aws:bedrock-agentcore:[^:]+:\d+:runtime/.+$")
    agents = {}

    for key, value in sorted(os.environ.items()):
        if not value or not arn_pattern.match(value):
            continue

        name = key.lower()
        for suffix in ["_runtime_arn", "_arn"]:
            if name.endswith(suffix):
                name = name[: -len(suffix)]
                break

        # Look for companion env vars (bearer token, runtime ID)
        prefix = key.rsplit("_ARN", 1)[0] if "_ARN" in key else key
        bearer_token = (
            os.getenv(f"{prefix}_BEARER_TOKEN")
            or os.getenv(f"{key.replace('_ARN', '_BEARER_TOKEN')}")
            or None
        )
        runtime_id = (
            os.getenv(f"{prefix}_ID")
            or os.getenv(f"{key.replace('_ARN', '_ID')}")
            or ""
        )

        agents[name] = {
            "arn": value,
            "runtime_id": runtime_id,
            "bearer_token": bearer_token or None,
            "env_var": key,
        }

    return agents

AGENTS = discover_agents()

if not AGENTS:
    print("⚠ No AgentCore agents found in environment.")
    print("  Add env vars with AgentCore runtime ARN values to .env.agentcore")
    print("  Example: MY_AGENT_RUNTIME_ARN=arn:aws:bedrock-agentcore:us-east-1:123456:runtime/abc")
else:
    print(f"✓ Discovered agent runtime:\n")
    for name, cfg in AGENTS.items():
        print(f"  {name}")
        print(f"    Source: ${cfg['env_var']}")
        print(f"    ARN:    {cfg['arn']}")
        print()


#### ONLY run the following cells if you have AgentCore runtime configured with JWT authentication (bearer token): 
Skip these cells if you're using SigV4 authentication.

In [ ]:
import boto3

def get_cognito_token(user_pool_id, client_id, username, password):
    """Authenticate with Cognito and return an access token."""
    client = boto3.client("cognito-idp", region_name="us-east-1")
    response = client.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={
            "USERNAME": username,
            "PASSWORD": password,
        },
    )
    return response["AuthenticationResult"]["AccessToken"]


Retrieve your Bearer Token using Cognito credentials:

In [ ]:
os.environ["RUNTIME_BEARER_TOKEN"] = get_cognito_token(
    user_pool_id="<enter-your-cognito-user-pool-id>",
    client_id="<enter-your-client-ID>",
    username="<enter-your-username>",
    password="<enter-your-password>",
)


Set up bearer token in your `.env` file:

In [ ]:
from dotenv import set_key

# Write to .env file (creates or updates the RUNTIME_BEARER_TOKEN line)
set_key(".env", "RUNTIME_BEARER_TOKEN", os.environ["RUNTIME_BEARER_TOKEN"])
RUNTIME_BEARER_TOKEN = os.environ["RUNTIME_BEARER_TOKEN"]
print("✓ Bearer token written to .env and loaded into environment")

### Proceed with the following cells for **both** SigV4 & JWT authentication:

### Pass a single query through AgentCore adapter

Confirm Runtime ARN and region:

In [ ]:
runtime = cfg['arn']
print(f"AgentCore Runtime ARN for multi-agent solution:\n\n {runtime})")

region = "us-east-1"
print(f"\n\nAWS Region: {region}")

Configure AgentCore adapter with UAEF

In [ ]:
adapter = AgentCoreAdapter()
print("UAEF AgentCore adapter configured.")


#### Invoke multi-agent solution

In [ ]:
user_input = "Compare the trend direction for the period ending 2022-10-14 versus the period ending 2023-02-02. Which period had a higher 20-day SMA?"

# Invoke agent 
raw_data = adapter.invoke(
    agent_runtime_arn=runtime,
    user_input=user_input,
    agent_version="DEFAULT",
    log_stream_name="otel-rt-logs",
    region=region,
    # bearer_token=RUNTIME_BEARER_TOKEN, # only enable when using JWT auth
)


print(f"✓ Agent responded in {raw_data['latency']:.2f}s")
print(f"\nAgent Response: {raw_data['final_response'][:500]}")

#### Transform trace with adapter

In [ ]:
agent_trace = adapter.transform_to_canonical(raw_data)
print(f"✓ Transformed AgentCore output to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")

if agent_trace.input_tokens:
    print(f"  Input Tokens: {agent_trace.input_tokens}")
if agent_trace.output_tokens:
    print(f"  Output Tokens: {agent_trace.output_tokens}")


# Print extracted messages
for msg in agent_trace.messages:
    print(f"\n  [{msg.role.value}]: {msg.content[:200]}")


# Identify subagents
if isinstance(agent_trace, MultiAgentTrace):
    print("\n subagents identified from trace:", list(agent_trace.agent_traces.keys()))
else:
    print("\n Single AgentTrace returned (no sub-agent detection from logs)")
    print(f"  To enable multi-agent detection, pass agent_version to invoke()")

#### Evaluate single query with Ground Truth

In [ ]:
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime, timezone

# Define Ground Truth for single query
ground_truth = GroundTruth(
    expected_output="Trend direction: downtrend vs uptrend. 20-day SMA: 116.14 vs 96.58...",
    expected_tool_calls=[
        ToolCall(name="detect_trend", arguments={"end_date": "2022-10-14"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="detect_trend", arguments={"end_date": "2023-02-02"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="calculate_moving_average", arguments={"end_date": "2022-10-14"}, timestamp=datetime.now(timezone.utc)),
        ToolCall(name="calculate_moving_average", arguments={"end_date": "2023-02-02"}, timestamp=datetime.now(timezone.utc)),
    ],
)

# Run evaluation for single query
result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics,
)
print(f"\n{'='*50}")
print("AGENTCORE — EVALUATION RESULTS")
print(f"{'='*50}")
print(f"Overall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

print(f"\n{'='*50}")
print("METRIC SCORES BY DIMENSION")

for dimension in result.dimension_results:
    print(f"\n{dimension.dimension_name} (aggregate score: {dimension.aggregate_score:.2f}):")
    for metric in dimension.metric_scores:
        if metric.score is not None:
            print(f"  {metric.metric_name}: {metric.score:.2f}")
        else:
            print(f"  {metric.metric_name}: N/A (failed)")


## Sample Multiagent - Batch Evaluation with Ground Truth

#### Load Ground Truth CSV for batch evaluation 

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime, timezone
from uuid import uuid4

from uaef.models import GroundTruth, ToolCall
from uaef.models.message import MessageRole
from uaef.evaluation.multi_agent import MultiAgentEvaluator

# Load ground truth data from Excel
excel_path = "data/multiagent_agentcore_ground_truth.csv"
df_gt = pd.read_csv(excel_path)
gt_json = df_gt.to_dict(orient="records")

print(f"✓ Loaded {len(gt_json)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df_gt.columns)}")
df_gt.head()


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import pandas as pd
from datetime import datetime
from uaef.adapters.agentcore import AgentCoreAdapter
from uaef.api import evaluate
from uaef.models.ground_truth import GroundTruth
from uaef.models.message import MessageRole

adapter = AgentCoreAdapter()
results = []
traces = []


### Invoke, Transform, and Evaluate with Multiple Queries

In [ ]:


for idx, row in df_gt.iterrows():
    query = row["question"]
    expected_output = row["expected_output"]
    print(f"\n[{idx+1}/{len(df_gt)}] {query[:80]}...")

    # Parse expected_tool_calls from CSV into ToolCall objects
    expected_tool_calls = []
    if pd.notna(row.get("expected_tool_calls", "")):
        try:
            # If stored as JSON string in CSV
            raw_tools = json.loads(str(row["expected_tool_calls"]))
            if isinstance(raw_tools, list):
                for t in raw_tools:
                    if isinstance(t, dict):
                        expected_tool_calls.append(ToolCall(
                            name=t.get("name", ""),
                            arguments=t.get("arguments", t.get("args", t.get("input", {}))),
                            timestamp=datetime.now(timezone.utc),
                        ))
                    elif isinstance(t, str):
                        # Just tool names
                        expected_tool_calls.append(ToolCall(name=t, arguments={}))
        except (json.JSONDecodeError, TypeError):
            # Fallback: comma-separated tool names
            for name in str(row["expected_tool_calls"]).split(","):
                name = name.strip()
                if name:
                    expected_tool_calls.append(ToolCall(name=name, arguments={}, timestamp=datetime.now(timezone.utc)))

    # 1. Invoke
    raw_data = AgentCoreAdapter.invoke(
        agent_runtime_arn=runtime,
        user_input=query,
        agent_version="DEFAULT",
        log_stream_name="otel-rt-logs",
        region=region,
        # bearer_token=RUNTIME_BEARER_TOKEN, # only enable when using JWT auth
    )

    # 2. Transform
    agent_trace = adapter.transform_to_canonical(raw_data)
    traces.append(agent_trace)

    # 3. Evaluate
    ground_truth = GroundTruth(expected_output=expected_output, expected_tool_calls=expected_tool_calls)
    result = evaluate(
        trace=agent_trace,
        ground_truth=ground_truth,
        metrics=metrics,
    )
    results.append(result)

    print(f"  Score: {result.overall_score:.2f} | Tools: {len(agent_trace.tool_calls)}")


#### Export batch evaluation to CSV

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="agentcore_batch_results")
